# 06 · Evaluation ground truth (KITTI-360 val and test drives)

Builds the BEV ground truth every metric is measured against, from **ground-truth data only**: the accumulated semantic point clouds, the 3D boxes, the poses and the calibration. It never uses the LiDAR scan or the student's geometry (see `ocgf/evaluation/gt_builder.py`).

Per frame (one `.npz`, 200 x 160 cells of 0.25 m): ground height, majority ground class, labelled, occupied (every annotated box, by class), obstacle (points 0.3-3 m above ground), observed from the LiDAR and from the camera (line of sight over ground-truth occluders). Free road = road, not occupied, no obstacle, labelled and observed.

Frames: the usable frames sampled every 5 m inside a semantic window, as in the paper split `kitti360_paper_v1` (val 0006, test 0009 and 0010).

## Before you run

1. **Settings → Accelerator:** None (CPU). **Internet:** On (code clone).
2. **Add Input:** the `kitti360-d<drive>-supplement` of each drive in `DRIVES`, the KITTI-360 starter (calibration), `data-poses`, `data-3d-bboxes` and **KITTI-360 3d semantics** (`greatgamedota/kitti360-3d-semantics`).
3. **Save Version → Save & Run All** (about 1 s per frame, roughly 1-2 h for the three drives). Drives without an attached supplement are skipped and listed.

In [ ]:
from pathlib import Path

REPO_URL  = "https://github.com/GNaveenkumar614/OCGF.git"
GIT_REF   = "p0/skeleton-geometry"       # branch, tag or commit (main once merged); pin a commit for the final run
DRIVES    = ["0006", "0009", "0010"]     # val and test drives of kitti360_paper_v1
INPUT_ROOT = Path("/kaggle/input")
OUT_DIR   = Path("/kaggle/working/eval_gt")      # kept as the notebook output
STAGE_DIR = Path("/kaggle/temp/semantics")       # local copy of the semantic windows
THREADS   = 16                                   # parallel readers: /kaggle/input is slow per file
QA        = 6                                    # QA figures per drive
ALL_FRAMES = False                               # True: every stored usable frame, not one every 5 m
CODE_DIR  = Path("/kaggle/temp/OCGF")

## Code

A fresh clone of the repo at `GIT_REF`, installed in editable mode.

In [ ]:
import os, shutil, subprocess, sys

def sh(cmd, cwd=None):
    # run a command, stream its output into this notebook, and stop Run All if it fails
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")

CODE_DIR.parent.mkdir(parents=True, exist_ok=True)
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)            # fresh clone, so the run matches GIT_REF exactly
sh(f'git clone -q {REPO_URL} "{CODE_DIR}"')
sh(f'git -C "{CODE_DIR}" checkout -q {GIT_REF}')
sh(f'git -C "{CODE_DIR}" log -1 --date=short --format="code at commit %h (%cd): %s"')
sh(f'{sys.executable} -m pip install -q -e "{CODE_DIR}"')
sys.path.insert(0, str(CODE_DIR))
for name in [m for m in sys.modules if m == "ocgf" or m.startswith("ocgf.")]:
    del sys.modules[name]

from ocgf.utils.provenance import environment
print(environment())

## Inputs

Each drive needs its data-prep output and the raw inputs it points to, including the accumulated 3D semantics.

In [ ]:
from ocgf.datasets.locate import prepared_drive

ready = []
for d in DRIVES:
    seq = f"2013_05_28_drive_{d}_sync"
    try:
        parts = prepared_drive(INPUT_ROOT, seq)
    except FileNotFoundError as err:
        print(f"drive {d}: skipped, {err}")
        continue
    if "semantics3d" not in parts:
        print(f"drive {d}: skipped, no accumulated 3D semantics found (attach KITTI-360 3d semantics)")
        continue
    print(f"drive {d}: ready, semantics in {parts['semantics3d']}")
    ready.append(seq)
assert ready, "no drive is ready: attach the drive supplements and the 3D semantics (Add Input)"


## Build

One drive at a time: the semantic windows are copied to local disk first, then every frame's layers are written.

In [ ]:
for seq in ready:
    sh(f'{sys.executable} scripts/build_eval_gt.py --input-root "{INPUT_ROOT}" --sequence {seq} --out "{OUT_DIR}" '
       f'--qa {QA} --stage-dir "{STAGE_DIR}" --threads {THREADS}' + (" --all-frames" if ALL_FRAMES else ""), cwd=CODE_DIR)

## Summary and QA figures

Mean share of the 50 x 40 m grid per layer, per drive, and two QA figures each: ground truth drawn on the image at ground level (it must sit on the road surface), and the BEV layers.

In [ ]:
import json
from IPython.display import Image, display

for seq in ready:
    manifest = json.loads((OUT_DIR / seq / "manifest.json").read_text())
    print(f"{seq}: {manifest['frames']} frames in {len(manifest['windows'])} windows; mean shares: {manifest['mean_shares']}")
    for png in sorted((OUT_DIR / seq / "qa").glob("*.png"))[:2]:
        display(Image(filename=str(png)))
sh(f'du -sh "{OUT_DIR}"/*')

## After the run

Save the output as the private dataset **`ocgf-eval-gt-v1`** (Output → New Dataset). The evaluation notebooks attach it; it is ground truth, never a training input.